# QF 627 Programming and Computational Finance
### Lesson 9 | Feature Extraction and Price Prediction (How to Integrate Unsupervised Learning into Supervised Learning)

### Activation of necessary modules

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib as mpl

from pandas_datareader import data as pdr

import datetime as dt
import yfinance as yf

In [ ]:
import warnings
warnings.filterwarnings("ignore")

#### Setting plotting and display options

In [ ]:
np.set_printoptions(precision = 3)

plt.style.use("ggplot")

mpl.rcParams["axes.grid"] = True
mpl.rcParams["grid.color"] = "grey"
mpl.rcParams["grid.alpha"] = 0.25

mpl.rcParams["axes.facecolor"] = "white"

mpl.rcParams["legend.fontsize"] = 14

In [ ]:
%matplotlib inline

## An ML Question

    PROBLEM STATEMENT: Our problem statement is how to use the dimensionality reduction approach to enhance 
    the bitcoin trading strategy.

    The data and the variables in this case study are the same as those in our previous task. The data is 
    the bitcoin data, with minute-by-minute updates of OHLC (Open, High, Low, Close), Volume in BTC, 
    and indicated currency and weighted bitcoin price.

    With dimensionality reduction, we achieved almost the same accuracy but a four times’ improvement in the time. In trading strategy development, when the datasets are huge and the features are many, such improvement in time can lead to a significant improvement in the entire process. 

### Load additional packages

In [ ]:
from pandas import read_csv, set_option
from pandas.plotting import scatter_matrix

import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier

from mpl_toolkits.mplot3d import Axes3D

import re
from collections import OrderedDict
from time import time
import sqlite3       

In [ ]:
from scipy.linalg import svd   
from scipy import stats
from sklearn.decomposition import TruncatedSVD
from sklearn.manifold import TSNE

In [ ]:
import ipywidgets as widgets
from ipywidgets import interact, interact_manual, fixed

### IMPORT

In [ ]:
# dataset =\
# (
#     pd
#     .read_csv("bitcoin.csv")
# )

### EDA

In [ ]:
# # shape
# dataset.shape

In [ ]:
# # peek at data
# set_option("display.width", 100)
# dataset.tail(5)

### Data Wrangle

In [ ]:
# print("Missing Values? ", dataset.isnull().values.any()
#      )

Given that there are null values, we need to clean the data by filling the *NaNs* with the last available values. 

In [ ]:
# dataset[dataset.columns.values] =\
#     dataset[dataset.columns.values].ffill()

In [ ]:
# dataset =\
# (
#     dataset
#     .drop(columns=["Timestamp"]
#          )
# )

#### Prepare Your Y (Outcome)

#### based on Price Movement

* `0` if the signal is that short-term (10) price will go up as compared to the long term (60)
* `1` if the signal is that short-term (10) price will go down as compared to the long term (60)

In [ ]:
# dataset["short_term"] =\
# (    
#     dataset
#     ["Close"]
#     .rolling(window = 10,
#              min_periods = 1,
#              center = False)
#     .mean()
# )

# dataset["long_term"] =\
# (
#     dataset
#     ["Close"]
#     .rolling(window = 60,
#              min_periods = 1,
#              center = False)
#     .mean()
# )

# # Let's set signals

# dataset["signal"] =\
# (    
#     np
#     .where(dataset["short_term"] > dataset["long_term"],
#            1.0, 0.0)
# )

### Feature Engineering

* Moving Average

* Relative Strength Index (`RSI`)

* Stochastic Oscillator %K & %D

* Momentum 

* Rate of Change (RoC)


#### We begin by constructing a dataset that contains the predictors which will be used to make the predictions, and the output variable.

#### The current bitcoin data includes date, open, high, low, close, and volume. Using this data, we calculate the following  technical indicators:

* `Moving Average`: A moving average indicates how the price is trending by reducing the “noise” on a price chart. 
<br>

* `Stochastic Oscillator %K and %D`: A stochastic oscillator is a momentum indicator that compares a security’s closing price to its prices over a certain time range. %K and %D are slow and fast indicators.
<br>

* `Relative Strength Index (RSI)`: RSI is a momentum indicator that measures the magnitude of recent price changes to evaluate overbought or oversold conditions in the price of a stock or other asset. 
<br>

* `Rate of Change(ROC)`: ROC is a momentum oscillator, which measures the percentage change between the current price and the n period past price. 
<br>

* `Momentum (MOM)`: MOM is the rate at which a security’s price or volume accelerates; that is, the speed at which the price is changing.


In [ ]:
# def MA(df, n):
#     MA = pd.Series(df["Close"]
#                    .rolling(n, min_periods = n)
#                    .mean(), 
#                    name="MA_" + str(n)
#                   )
#     return MA

In [ ]:
# dataset["MA21"] = MA(dataset, 10)
# dataset["MA63"] = MA(dataset, 30)
# dataset["MA252"] = MA(dataset, 200)

# dataset

In [ ]:
# # Exponential Moving Average

# def EMA(df, n):
#     EMA = pd.Series(df["Close"]
#                     .ewm(span = n,
#                          min_periods = n)
#                     .mean(),
#                     name = "EMA_" + str(n)
#                     )
#     return EMA

In [ ]:
# dataset["EMA10"] = EMA(dataset, 10)
# dataset["EMA30"] = EMA(dataset, 30)
# dataset["EMA200"] = EMA(dataset, 200)

In [ ]:
# dataset

In [ ]:
# # Rate of Change (RoC)

# def ROC(df, n):
#     M = df.diff(n - 1)
#     N = df.shift(n - 1)
#     ROC = pd.Series((M / N) * 100, 
#                     name = "ROC_" + str(n)    
#                     )
#     return ROC

In [ ]:
# dataset["ROC10"] = ROC(dataset["Close"], 10)
# dataset["ROC30"] = ROC(dataset["Close"], 30)

# dataset

In [ ]:
# # Price Momentum

# def MOM(df, n):
#     MOM = pd.Series(df.diff(n),
#                     name = "MOM_" + str(n)
#                    )
#     return MOM

In [ ]:
# dataset["MOM10"] = MOM(dataset["Close"], 10)
# dataset["MOM30"] = MOM(dataset["Close"], 30)

In [ ]:
# dataset

### Relative Strength Index (RSI)

In [ ]:
# def RSI(series, period):
    
#     delta = series.diff().dropna()
    
#     u = delta * 0
#     d = u.copy()
    
#     u[delta > 0] = delta[delta > 0]
#     d[delta < 0] = -delta[delta < 0]
    
#     u[u.index[period - 1]] = np.mean( u[:period] ) # 
    
#     u = u.drop(u.index[:(period - 1)
#                       ]
#               )
    
#     d[d.index[period - 1]] = np.mean( d[:period] )
    
#     d = d.drop(d.index[:(period - 1)
#                       ]
#               )
    
#     rs = u.ewm(com = period - 1, adjust = False).mean() / \
#          d.ewm(com = period - 1, adjust = False).mean()
    
#     return 100 - 100 / (1 + rs)

In [ ]:
# dataset["RSI10"] = RSI(dataset["Close"], 10)
# dataset["RSI30"] = RSI(dataset["Close"], 30)
# dataset["RSI200"] = RSI(dataset["Close"], 200)

In [ ]:
dataset

### Stochastic Oscillator 

In [ ]:
# def STOK(close, low, high, n):
#     STOK = ((close - low.rolling(n).min()) / \
#             (high.rolling(n).max() - low.rolling(n).min()
#             )
#            ) * 100
#     return STOK

# def STOD(close, low, high, n):
#     STOK = ((close - low.rolling(n).min()) / \
#             (high.rolling(n).max() - low.rolling(n).min()
#             )
#            ) * 100
    
#     STOD = STOK.rolling(3).mean()
#     return STOD

In [ ]:
# dataset["%K10"] = STOK(dataset["Close"], dataset["Low"], dataset["High"], 10)
# dataset["%D10"] = STOD(dataset["Close"], dataset["Low"], dataset["High"], 10)

# dataset["%K30"] = STOK(dataset["Close"], dataset["Low"], dataset["High"], 30)
# dataset["%D30"] = STOD(dataset["Close"], dataset["Low"], dataset["High"], 30)

# dataset["%K200"] = STOK(dataset["Close"], dataset["Low"], dataset["High"], 200)
# dataset["%D200"] = STOD(dataset["Close"], dataset["Low"], dataset["High"], 200)

In [ ]:
# dataset.describe()

#### Drop the features that are not include in your algorithms

In [ ]:
# dataset =\
# (
#     dataset
#     .drop(["Open", "High", "Low", "Volume_(Currency)", "short_term", "long_term"],
#           axis = 1)
# ) # note the axis )

In [ ]:
# dataset = dataset.dropna(axis = 0) # note the axis

In [ ]:
# dataset.info()

### Exploratory Data Analysis (`EDA`)

In [ ]:
# dataset["Weighted_Price"].plot()

# plt.show()

In [ ]:
# dataset \
#     .hist(sharex = False,
#           sharey = False,
#           xlabelsize = 1,
#           ylabelsize = 1,
#           figsize = (16, 16)
#           )

# plt.show()

In [ ]:
# # Correlation matrix
# correlation = dataset.corr()

# plt.figure(figsize=(15,15))
# plt.title("Correlation Matrix")

# sns.heatmap(correlation, 
#             vmax = 1, 
#             square = True,
#             annot = True,
#             cmap = "viridis")

In [ ]:
# dataset[["Weighted_Price"]].plot(grid = True)

# plt.show()

In [ ]:
# fig = plt.figure(figsize = (10, 8)
#                 )
# plot = dataset.groupby(["signal"]).size().plot(kind = "barh", 
#                                                color = "purple")
# plt.show()

The predicted variable is upward 52.87% out of total data-size, meaning that number
of the buy signals was higher than that of sell signals.

## MODEL

### Data Split

> We split the dataset into 80% training set and 20% test set.

In [ ]:
# subset_dataset= dataset.iloc[-10000:]

In [ ]:
# Y= subset_dataset["signal"]
# X = subset_dataset.loc[:, dataset.columns != "signal"]
# validation_size = 0.2
# seed = 1

# X_train, X_validation, Y_train, Y_validation =\
#     train_test_split(X, Y, test_size = validation_size, random_state = 1)

### Feature Extraction

> As a preprocessing step, let's start with normalizing the feature values so they standardised - this makes comparisons simpler and allows next steps for Singular Value Decomposition.

In [ ]:
# from sklearn.preprocessing import StandardScaler

In [ ]:
# scaler = StandardScaler().fit(X_train)
# rescaledDataset = pd.DataFrame(scaler.fit_transform(X_train),columns = X_train.columns, index = X_train.index)

In [ ]:
# # summarize transformed data
# X_train.dropna(how = "any", inplace = True)
# rescaledDataset.dropna(how = "any", inplace = True)
# rescaledDataset.head(2)

### Singular Value Decomposition for Feature Extraction

We want to reduce the dimensionality of the problem to make it more manageable, but at the same time we want to preserve as much information as possible. 

Hence, we use a technique called singu‐
lar value decomposition (SVD), which is one of the ways of performing PCA.Singular Value Decomposition (SVD) is a matrix factorization commonly used in signal processing and data compression. We are using the TruncatedSVD method in the sklearn package.

In [ ]:
# from matplotlib.ticker import MaxNLocator

In [ ]:
# ncomps = 5

# svd = TruncatedSVD(n_components=ncomps)

# svd_fit = svd.fit(rescaledDataset)

# plt_data = pd.DataFrame(svd_fit.explained_variance_ratio_.cumsum()*100)

# plt_data.index = np.arange(1, len(plt_data) + 1)

# Y_pred = svd.fit_transform(rescaledDataset)

# ax = plt_data.plot(kind="line", figsize=(20, 10), style = "o-"
#                   )
# ax.xaxis.set_major_locator(MaxNLocator(integer = True)
#                           )

# ax.set_xlabel("PCs")
# ax.set_ylabel("Percentage Explained")
# ax.legend("")

# print("Variance preserved by first 5 components == {:.2%}".format(svd_fit.explained_variance_ratio_.cumsum()[-1]
#                                                                  )
#      )

We can preserve 92.75% variance by using just 5 components rather than the full 25+ original features.

In [ ]:
# dfsvd = pd.DataFrame(Y_pred, columns=["pc{}".format(c) for c in range(ncomps)], 
#                      index = rescaledDataset.index)
# print(dfsvd.shape)
# dfsvd.head()

## Compare Models with and without Dimensionality Reduction

In [ ]:
# # test options for classification
# scoring = "accuracy"
# seed = 627
# kfold = 10

### Model without Dimensionality Reduction

In [ ]:
# dfsvd

In [ ]:
# print("Result without dimensionality Reduction: %f (%f)" % (cv_results_XTrain.mean(), cv_results_XTrain.std()
#                                                            )
#      )
# print("Result with dimensionality Reduction (Yay~): %f (%f)" % (cv_results_SVD.mean(), cv_results_SVD.std()
#                                                         )
#      )

## Things to Note

> In terms of the evaluation metrics for a classification-based trading strategy, accuracy or AUC are appropriate. But if the strategy required is greater focus on accuracy while going long, the metric recall which focuses on fewer false positives is preferable to a focus on accuracy.

> Finally, we demonstrated the backtesting framework which allows us to simulate a trading strategy using historical data to generate results and analyse risk and profitability before risking any actual capital.

> We demonstrated that both SVD and t-SNE provide quite an interesting way of visualizing the trading signal data, and provide a way to distinguish long and short signals of a trading strategy with a reduced number of features.

> `Thank you for working with the script, Team 👍`